# CellTransformer2 on Allen 1 MERFISH: full run with the native commands

This notebook trains CellTransformer2 on the complete Allen 1 dataset (MERFISH-C57BL6J-638850, all
sections) with the same native commands as `03_custom_dataset.ipynb`:
`celltransformer-preprocess` → `celltransformer-train` → `celltransformer-extract` → `celltransformer-smooth` → `celltransformer-cluster`.
The steps and cells mirror notebook 03, so what you learn here carries over to your own data.
The only differences are the data (all sections instead of five) and the design settings, which match
the original Allen example (85 µm neighborhoods, 10 epochs).

**Before you start:**
- About 10 GB of storage for the raw Allen files, which Step 1 downloads and checksums into
  `DATA_ROOT/data/raw/`, and roughly 30 GB for everything this notebook writes to `WORK_DIR`.
- Two environments, both with the package installed (`pip install -e .` from the repository):
  a training environment (this notebook's kernel, with `torchdata>=0.11`), and a RAPIDS environment
  with cuML, used for smoothing and clustering.
- Linux x86_64 and an NVIDIA GPU with bfloat16 support.
- Notebook support and HTML reporting are included in the installed CellTransformer2 package.

**Input contract.** Every later step reads the standard input written in Step 3, one pair per section:
an `.h5ad` of raw counts (cells × genes) in `layers['raw']` and a paired `.parquet` of the same cells, in the same order,
with columns `section`, `x`, `y` (µm) and `cell_type`. Row *i* of the metadata must describe row *i* of
the count matrix.

## Step 1: set paths and dataset columns
● TIMING <1 min, plus the first download (~9.3 GB)

Set `DATA_ROOT`, `WORK_DIR` and `RAPIDS_BIN`, then run the next cell to download the raw Allen files
(files already downloaded and verified are skipped). The column names below are those of the Allen metadata.
All sections are used. Validation sections are chosen in Step 6.

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import yaml

from celltransformer.notebooks import helper

#where the raw Allen files go (the next cell downloads them into DATA_ROOT/data/raw/)

DATA_ROOT = Path("~/allen_data").expanduser().resolve()
RAW_H5AD = DATA_ROOT / "data/raw/expression_raw.h5ad"
RAW_METADATA = DATA_ROOT / "data/raw/cells_ccf.csv"

#metadata column names
CELL_ID_COL = "cell_label"            #matches the .h5ad obs_names
SECTION_COL = "brain_section_label"
CELL_TYPE_COL = "subclass"
X_COL = "x_reconstructed"
Y_COL = "y_reconstructed"
COORD_TO_UM = 1000.0                  #multiply coordinates by this to get µm (Allen stores mm)

#genes to drop (MERFISH blank control probes), matched on this var column
GENE_SYMBOL_COL = "gene_symbol"
DROP_GENE_PREFIX = "Blank"

#working directory for everything this notebook writes
WORK_DIR = Path("~/ct2_runs/allen_full").expanduser().resolve()

#bin/ directories of the two environments: this kernel's, and the RAPIDS one (set this)
DEV_BIN = Path(sys.executable).parent
#REPLACE WITH YOUR ENV: bin/ of celltransformer2-rapids; run `conda env list` and add /bin to its path
RAPIDS_BIN = Path("/path/to/rapids-env/bin").resolve()

#derived paths
INPUT_DIR = WORK_DIR / "input"
SECTIONS_DIR = WORK_DIR / "sections"
SMOOTHED_DIR = WORK_DIR / "sections_smoothed"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
DOMAINS_DIR = WORK_DIR / "domains"
TAXONOMY = WORK_DIR / "taxonomy.yaml"
TRAIN_LOG = WORK_DIR / "train.log"
for d in [INPUT_DIR, SECTIONS_DIR, CHECKPOINT_DIR, DOMAINS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
%%time
#download the raw Allen files (~9.3 GB) and verify their checksums; verified files are skipped
helper.download_allen(DATA_ROOT)

## Step 2: check the environment
● TIMING ~1 min

Lists the visible GPUs, checks that PyTorch can use the GPU in this environment, and checks that the
RAPIDS environment in `RAPIDS_BIN` has the smoothing and clustering commands and a working cuML/cuDF.
Do not continue until it reports `ENVIRONMENT CHECK: PASSED`.

In [ ]:
%%time
helper.check_environment(DEV_BIN, RAPIDS_BIN)

## Step 3: write the standard input
● TIMING ~1 min for the full dataset (reading and writing ~3.7 million cells)

For the Allen data this step:
1. writes the cell-type taxonomy from the full metadata,
2. keeps annotated cells,
3. matches metadata rows to count rows by exact cell ID,
4. drops blank control probes, and
5. converts coordinates to µm.

▲ **CRITICAL STEP** The taxonomy is an ordered list; a cell type's position in it is its integer label.
`celltransformer-train` and `celltransformer-extract` must use the same file.

▲ **CRITICAL STEP** Keep raw counts. `celltransformer-preprocess` applies log2(1 + x) in Step 4; do not
normalize or log-transform beforehand.

The counts are read and written one section at a time (`input/{section}.h5ad` + `.parquet`), so only
one section's counts are in memory; the metadata for all cells stays loaded. Rerunning the second cell
skips sections already written, so the step can continue after a crash.

In [ ]:
%%time
#load the metadata columns this notebook uses
columns = [CELL_ID_COL, SECTION_COL, CELL_TYPE_COL, X_COL, Y_COL]
if RAW_METADATA.suffix == ".parquet":
    metadata = pd.read_parquet(RAW_METADATA, columns=columns)
else:
    metadata = pd.read_csv(RAW_METADATA, usecols=columns, dtype={CELL_ID_COL: str})

#taxonomy from the full dataset
taxonomy = sorted(str(t) for t in metadata[CELL_TYPE_COL].dropna().unique())
TAXONOMY.write_text(yaml.safe_dump(taxonomy, allow_unicode=True))
print(f"taxonomy: {len(taxonomy)} cell types -> {TAXONOMY}")

#keep annotated cells (all sections)
metadata = metadata.dropna(subset=[CELL_TYPE_COL])

#match metadata rows to count rows by exact cell ID (backed: only cell IDs and gene names are loaded)
counts = ad.read_h5ad(RAW_H5AD, backed="r")
count_rows = pd.Index(counts.obs_names).get_indexer(metadata[CELL_ID_COL])
has_counts = count_rows >= 0
print(f"{has_counts.sum():,} annotated cells with counts; {(~has_counts).sum():,} without counts dropped")
metadata = metadata[has_counts]
count_rows = count_rows[has_counts]

#file order (required for backed reads), keeping metadata aligned; each section's rows stay in file order
order = np.argsort(count_rows)
metadata = metadata.iloc[order].reset_index(drop=True)
count_rows = count_rows[order]

#control probes to drop
keep_genes = ~counts.var[GENE_SYMBOL_COL].astype(str).str.startswith(DROP_GENE_PREFIX).to_numpy()
genes = counts.var[keep_genes]

#standard metadata: section, x, y (µm), cell_type, in count-matrix row order
standard_metadata = pd.DataFrame({
    "cell_id": metadata[CELL_ID_COL].astype(str).to_numpy(),
    "section": metadata[SECTION_COL].astype(str).to_numpy(),
    "x": metadata[X_COL].to_numpy() * COORD_TO_UM,
    "y": metadata[Y_COL].to_numpy() * COORD_TO_UM,
    "cell_type": metadata[CELL_TYPE_COL].astype(str).to_numpy(),
})
print(f"{len(standard_metadata):,} cells x {len(genes)} genes in {standard_metadata['section'].nunique()} sections")

In [ ]:
%%time
#one input pair per section, reading only that section's counts
for section, rows in standard_metadata.groupby("section").indices.items():
    h5ad_path = INPUT_DIR / f"{helper.safe_section(section)}.h5ad"
    parquet_path = h5ad_path.with_suffix(".parquet")

    #the parquet is written last, so it marks a finished section (a rerun after a crash skips it)
    if parquet_path.exists():
        print("exists, skipping:", parquet_path.name)
        continue

    section_metadata = standard_metadata.iloc[rows].reset_index(drop=True)
    section_raw = counts[count_rows[rows]].to_memory()

    #row alignment check before writing
    assert (section_raw.obs_names == section_metadata["cell_id"]).all()

    #raw counts only in layers['raw'], which preprocess transforms into .X (so .X stays empty here)
    section_counts = ad.AnnData(
        obs=pd.DataFrame(index=section_metadata["cell_id"].to_numpy()),
        var=genes,
        layers={"raw": section_raw.X[:, keep_genes].astype(np.float32)},
    )
    section_counts.write_h5ad(h5ad_path)
    section_metadata.to_parquet(parquet_path, index=False)
    print(f"wrote {h5ad_path.name}: {len(section_metadata):,} cells")

counts.file.close()
standard_metadata.groupby("section").size()

## Step 4: log-transform the counts
● TIMING ~2 min for the full dataset

Step 3 is specific to the dataset; from here on, every step runs the same for any dataset.
`celltransformer-preprocess` runs once per section written in Step 3, so it loads one section at a time. It writes an
`.h5ad`/`.parquet` pair per section with log2(1 + x) of the raw counts in `.X`. Section names have `.` and
spaces replaced by `_` in the filenames. Sections already in `SECTIONS_DIR` are skipped.

In [ ]:
%%time
#one preprocess call per section (only that section in memory); finished sections are skipped
for parquet_path in sorted(INPUT_DIR.glob("*.parquet")):
    if (SECTIONS_DIR / f"allen_full_{parquet_path.stem}.parquet").exists():
        print("exists, skipping:", parquet_path.stem)
        continue
    helper.run("celltransformer-preprocess", parquet_path.with_suffix(".h5ad"),
        "--output-dir", SECTIONS_DIR,
        "--output-stem", "allen_full",
        "--section-col", "section",
        "--transform", "log2p1",
        cwd=WORK_DIR);

## Step 5: check the per-section files
● TIMING ~1 min

▲ **CRITICAL STEP** Every section must have as many metadata rows as expression rows, the same gene
panel, the required columns, only cell types present in the taxonomy, and log-scale expression.
Fix any failure here before training; the loader does not check these.

After a kernel restart, rerun Step 1 and then start here; Step 3 does not need rerunning.

In [ ]:
%%time
#taxonomy as train/extract read it (lets a restarted kernel skip Step 3)
taxonomy = yaml.safe_load(TAXONOMY.read_text())

def check_section(h5ad_path):
    '''Summarize one section pair and flag contract violations.'''
    expression = ad.read_h5ad(h5ad_path, backed="r")
    section_metadata = pd.read_parquet(h5ad_path.with_suffix(".parquet"))
    sample = expression.X[:1000]
    sample = sample.toarray() if hasattr(sample, "toarray") else np.asarray(sample)
    return {
        "file": h5ad_path.name,
        "cells": expression.n_obs,
        "genes": expression.n_vars,
        "rows_match": expression.n_obs == len(section_metadata),
        "columns_ok": {"section", "x", "y", "cell_type"} <= set(section_metadata.columns),
        "unknown_types": len(set(section_metadata["cell_type"]) - set(taxonomy)),
        "max_expression": float(sample.max()),
    }

section_files = sorted(SECTIONS_DIR.glob("*.h5ad"))
summary = pd.DataFrame([check_section(path) for path in section_files])

#all sections must pass
assert summary["rows_match"].all() and summary["columns_ok"].all(), "metadata/expression mismatch"
assert summary["genes"].nunique() == 1, "sections have different gene panels"
assert (summary["unknown_types"] == 0).all(), "cell types missing from the taxonomy"
assert summary["max_expression"].max() < 25, "expression looks untransformed (expected log2 scale)"
N_GENES = int(summary["genes"].iloc[0])
summary

## Step 6: choose the design
● TIMING <1 min

- **Neighborhood:** `PATCH_SIZE` is the side of the square window around each cell, in the same units as
  `x`/`y` (µm here). 85 µm follows the original CellTransformer paper.
- **Model size:** `MODEL_DIM` is the width, the length of the vector each cell becomes inside the
  transformer and of the final embedding. `NUM_HEADS` must divide it evenly.
- **Training:** `EPOCHS`, learning rate and warmup. `BATCH_SIZE` is centers per GPU and `MAX_TOKENS` caps
  the total neighbor cells per batch; lower `MAX_TOKENS` if you run out of GPU memory.
- **GPUs:** `DEVICES` lists GPU indices.
- **Validation sections:** whole sections are held out from training and used only for the validation
  loss. 6 sections are drawn at random with a fixed seed (42), so every run holds out the same ones.

The number of genes and cell types are set by the data (Steps 3 and 5), not chosen here.

In [ ]:
#neighborhood (coordinate units, µm here)
PATCH_SIZE = 85.0

#model
MODEL_DIM = 384
NUM_HEADS = 8
ENCODER_DEPTH = 4
DECODER_DEPTH = 4
FFN_DIM = 384
DROPOUT = 0.0

#training
EPOCHS = 10
LEARNING_RATE = 3e-4
WARMUP_STEPS = 500
BATCH_SIZE = 256
MAX_TOKENS = 16384
ACCUMULATE = 1
DEVICES = [0]                         #GPU indices

#validation sections: 6 drawn with seed 42 from the sorted section list (any 6 work)
N_VALID = 6
VALID_SEED = 42
all_sections = sorted(pd.read_parquet(p, columns=["section"])["section"].iloc[0] for p in INPUT_DIR.glob("*.parquet"))
rng = np.random.default_rng(VALID_SEED)
VALID_SECTIONS = sorted(rng.choice(all_sections, size=N_VALID, replace=False))
print(f"{len(all_sections)} sections; validation: {VALID_SECTIONS}")

#set by the data
N_TYPES = len(taxonomy)

assert MODEL_DIM % NUM_HEADS == 0, "NUM_HEADS must divide MODEL_DIM"
print(f"expression_dim={N_GENES}, cell_type_cardinality={N_TYPES}, head size={MODEL_DIM // NUM_HEADS}")

In [ ]:
neighbors = helper.neighbor_counts(section_files[0].with_suffix(".parquet"), PATCH_SIZE)
print(f"neighbors per cell at PATCH_SIZE={PATCH_SIZE}: median {np.median(neighbors):.0f}, "
      f"5th-95th percentile {np.percentile(neighbors, 5):.0f}-{np.percentile(neighbors, 95):.0f}, "
      f"{(neighbors == 0).mean():.1%} with none")

▲ **CRITICAL STEP** If most cells have zero neighbors, or thousands, `PATCH_SIZE` is probably in the
wrong units for your coordinates (for example mm vs µm). Nothing downstream checks this.

## Step 7: train
● TIMING ~5 h 20 min for 10 epochs on one GPU

`celltransformer-train` composes its configuration from `config/train_redone_registration.yaml` and the
`data`, `model` and `optimization` groups, then applies the overrides built below. Every value set here
replaces the default in those YAML files for this run only; the files themselves are not changed.
Training runs in the background, logging to `train.log`. W&B runs offline, so no account is needed.
If `checkpoints/` already has a finished run, this cell does nothing.

In [ ]:
#validation sections are excluded from training automatically
valid_patterns = ",".join(f'"*{helper.safe_section(s)}.h5ad"' for s in VALID_SECTIONS)

overrides = [
    #data
    f"data.data_dir='{SECTIONS_DIR}/'",
    'data.train_anndata_patterns=["*.h5ad"]',
    f"data.valid_anndata_patterns=[{valid_patterns}]",
    "data.cell_type_colname=cell_type",
    f"data.cell_type_yaml='{TAXONOMY}'",
    "data.tissue_section_colname=section",
    f"data.patch_size=[{PATCH_SIZE},{PATCH_SIZE}]",
    f"data.batch_size={BATCH_SIZE}",
    f"data.max_tokens={MAX_TOKENS}",
    #model
    f"model.expression_dim={N_GENES}",
    f"model.cell_type_cardinality={N_TYPES}",
    f"model.model_dim={MODEL_DIM}",
    f"model.num_heads={NUM_HEADS}",
    f"model.encoder_depth={ENCODER_DEPTH}",
    f"model.decoder_depth={DECODER_DEPTH}",
    f"model.ffn_dim={FFN_DIM}",
    f"model.dropout={DROPOUT}",
    #optimization
    f"optimization.lr={LEARNING_RATE}",
    f"optimization.schedule_warmup={WARMUP_STEPS}",
    f"optimization.max_epochs={EPOCHS}",
    f"optimization.accumulate_grad_batches={ACCUMULATE}",
    #machine and outputs
    f"devices=[{','.join(str(d) for d in DEVICES)}]",
    f"checkpoint_dir='{CHECKPOINT_DIR}'",
    "wandb_project=celltransformer2-allen-full",
    f"output='{WORK_DIR / 'train_results.jsonl'}'",
    f"hydra.run.dir='{WORK_DIR / 'hydra'}'",
]
print("celltransformer-train \\\n    " + " \\\n    ".join(overrides))

In [ ]:
helper.start_training(DEV_BIN, overrides, WORK_DIR, TRAIN_LOG, CHECKPOINT_DIR)

■ **PAUSE POINT** Training runs outside the notebook: it continues if you close the notebook, restart or shut
down the kernel, or disconnect from the machine. Rerun Steps 1, 5 and 6 when you come back,
then Step 8.

## Step 8: follow training
● TIMING <1 min (rerun as often as you like)

Shows whether training is alive and which finished run Step 9 will use. While training runs, or if no run has
finished yet, it also shows the end of the log. Each run gets a
timestamped folder under `checkpoints/`; `last.ckpt` is the final state.

▲ **CRITICAL STEP** Before extracting embeddings, the validation loss (`valid/nll`) should have stopped
decreasing. A loss still falling means more epochs are needed; a validation loss rising while training
loss falls means overfitting.

In [ ]:
helper.show_training(WORK_DIR, TRAIN_LOG, CHECKPOINT_DIR)

## Step 9: extract embeddings
● TIMING ~8 min for all 53 sections

Passes every cell's neighborhood through the trained encoder and writes one `*_embeddings.npz` per
section next to its `.h5ad`, with `embeddings` (cells × `MODEL_DIM`) and `has_neighbors`. Uses the most recent finished run.

▲ **CRITICAL STEP** `--patch-size` and the taxonomy must be the ones used for training. They are
reused from Steps 5 and 6 here; if you restart the notebook, rerun those cells first.

In [ ]:
%%time
run = helper.finished_run(CHECKPOINT_DIR)
assert run is not None, "training has not finished yet (no finished run in CHECKPOINT_DIR); see Step 8"
CHECKPOINT = run / "last.ckpt"
print("checkpoint:", CHECKPOINT)

for h5ad_path in section_files:
    helper.run("celltransformer-extract", h5ad_path,
        "--checkpoint-path", CHECKPOINT,
        "--cell-type-yaml", TAXONOMY,
        "--cell-type-colname", "cell_type",
        "--section-colname", "section",
        "--patch-size", PATCH_SIZE,
        "--device", f"cuda:{DEVICES[0]}",
        cwd=WORK_DIR, quiet=True)

## Step 10: smooth the embeddings spatially
● TIMING ~14 min for the full dataset

`celltransformer-smooth` replaces each cell's embedding with a Gaussian-weighted average of its
`N_NEIGHBORS` nearest neighbors in the same section (kernel width `SIGMA`, in µm). Sections are smoothed
independently, so no averaging crosses tissue boundaries. Runs in the RAPIDS environment, on the first GPU
in `DEVICES`.

It reads every `*_embeddings.npz` in `SECTIONS_DIR` and writes files of the same name, plus a copy of each
`.parquet`, to `SMOOTHED_DIR`. That folder has the same layout as `SECTIONS_DIR`, so Step 11 can cluster either.

▲ **CRITICAL STEP** `SIGMA` is in the same units as `x`/`y`. The value below matches the Allen example
(σ = 12.01 µm, k = 500); the command's own default (100) is a much wider blur.

In [ ]:
%%time
#smoothing kernel (coordinate units, µm here) and neighbors per cell
SIGMA = 12.01
N_NEIGHBORS = 500

helper.run("celltransformer-smooth", SECTIONS_DIR, SMOOTHED_DIR,
    "--gaussian-kernel", SIGMA,
    "--n-neighbors", N_NEIGHBORS,
    "--spatial-x-col", "x",
    "--spatial-y-col", "y",
    bin_dir=RAPIDS_BIN, cwd=WORK_DIR, gpu=DEVICES[0]);

## Step 11: cluster into spatial domains
● TIMING ~15 min for the full dataset

GPU k-means on all sections' embeddings together, once for each k in `KS`, first on the unsmoothed embeddings
(`SECTIONS_DIR`) and then on the smoothed ones (`SMOOTHED_DIR`). k is decoupled from training:
you can rerun this step with other values without retraining. Runs in the RAPIDS environment.

**Why these k values?** The Allen Common Coordinate Framework (CCF) atlas labels each cell at three
anatomical levels. On these sections there are 25 divisions (e.g. isocortex, hippocampus),
354 structures (e.g. primary motor cortex) and 670 substructures (e.g. its individual layers).
Clustering at k = 25, 354 and 670 gives one set of CellTransformer domains per level, so each can be
compared with the anatomical map at the same level of detail. For other data, choose k for the level
of detail you want; there is no single correct value.

The clustering runs on the first GPU in `DEVICES` (`CUDA_VISIBLE_DEVICES`), since
`celltransformer-cluster` otherwise uses GPU 0.

In [ ]:
%%time
KS = [25, 354, 670] #CCF divisions / structures / substructures

#output tag for each embedding set
EMBEDDING_SETS = {"allen_full": SECTIONS_DIR, "allen_full_smoothed": SMOOTHED_DIR}

k_args = [arg for k in KS for arg in ["--ks", k]]
for tag, embeddings_dir in EMBEDDING_SETS.items():
    helper.run("celltransformer-cluster",
        "--embeddings-dirs", embeddings_dir,
        "--output-base", DOMAINS_DIR,
        "--tag", tag,
        *k_args,
        "--x-col", "x",
        "--y-col", "y",
        "--section-col", "section",
        bin_dir=RAPIDS_BIN, cwd=WORK_DIR, gpu=DEVICES[0]);

## Step 12: view the domains
● TIMING <1 min

Labels are saved as `domains/{tag}_kmeans_k{k}_labels.npy` (one label per cell, in section-file order),
and one plot per section in `domains/{tag}_cluster_plots_k{k}/`, for `tag` = `allen_full` (unsmoothed) and
`allen_full_smoothed`. Colors are assigned independently in each run, so the same color does not mean the same
domain across plots.

In [ ]:
helper.show_domain_plots(DOMAINS_DIR, EMBEDDING_SETS, KS, n_show=1)

## Step 13: generate the HTML report (optional)
● TIMING ~1 min

Run this after Steps 9–11 have finished. It reads the saved assignments and writes a new,
self-contained `review.html` and `results.zip` under `WORK_DIR/html_reports/`. It does not retrain,
extract, smooth or cluster again. Training curves are recovered from the selected checkpoint's local
W&B history; missing values are reported rather than replaced with batch losses.

The report contains training diagnostics and maps for every section and k. If reference annotations
are supplied, it also shows reference–domain correspondence heatmaps: Pearson correlations between
cell-type proportions, with a fixed −1 to +1 color scale. Reference regions and discovered domains
need not have the same number of labels. The report has no separate composition-matrix panels.

The reporting functions are included in the installed CellTransformer2 package.
See the [reporting guide](../docs/notebook_html_report.md).
Section names come from the data. Set the report title, coordinate units and optional reference
columns below. To omit reference comparison, set `REFERENCE_COLUMNS = {}`. Reference rows are joined
by exact cell ID; a missing annotation excludes that cell from correspondence calculations, not from
training or the saved domain assignments. Colors are consistent across sections within each method
and k, but do not imply matched identities across methods or the reference.


In [ ]:
from celltransformer.notebooks.reporting import report_from_notebook, read_reference_table, read_training_history, show_report

REPORT_NAME = "Allen 1 MERFISH"
REPORT_COORD_UNITS = "µm"
REFERENCE_FILE = RAW_METADATA
REFERENCE_ID_COLUMN = CELL_ID_COL
REFERENCE_COLUMNS = {
    "CCF divisions": "parcellation_division",
    "CCF structures": "parcellation_structure",
    "CCF substructures": "parcellation_substructure",
}
REFERENCE_FOR_K = {k: name for k, name in {25: "CCF divisions", 354: "CCF structures", 670: "CCF substructures"}.items() if k in KS}
REFERENCE_DISPLAY_NAMES = {name: "CCF" for name in REFERENCE_COLUMNS}

# Optional matched cortical crops, in the same units as x/y.
REPORT_CROP = {
    "column": "parcellation_structure", "prefixes": ["MO", "SS"],
    "fallback_column": "parcellation_division", "fallback_values": ["Isocortex"],
    "min_cells": 100, "padding": 100,
}
REPORT_CROP_NAME = "Cortical enlargement"
REPORT_BOUNDARY_GRID = 20
REPORT_BOUNDARY_DISTANCE = 60


In [ ]:
# Read the completed checkpoint's local epoch averages; no W&B login is needed.
report_training = read_training_history(CHECKPOINT, expected_epochs=EPOCHS)
report_reference = read_reference_table(REFERENCE_FILE, REFERENCE_ID_COLUMN, REFERENCE_COLUMNS)

report_notes = report_training["notes"] + [
    f"{len(VALID_SECTIONS)} whole sections were withheld during training; clustering used all sections jointly.",
    "Reference correspondence is an exploratory comparison of cell-type composition, not an independent held-out clustering test.",
]
report_html = report_from_notebook(
    work_dir=WORK_DIR,
    embedding_sets=EMBEDDING_SETS,
    domains_dir=DOMAINS_DIR,
    ks=KS,
    dataset_name=REPORT_NAME,
    coordinate_units=REPORT_COORD_UNITS,
    n_genes=N_GENES,
    method_names={tag: "CellTransformer2 (+smoothing)" if Path(folder).resolve() == SMOOTHED_DIR.resolve()
                  else "CellTransformer2 (−smoothing)" for tag, folder in EMBEDDING_SETS.items()},
    reference_metadata=report_reference,
    reference_id_column=REFERENCE_ID_COLUMN,
    reference_columns=REFERENCE_COLUMNS,
    reference_for_k=REFERENCE_FOR_K if report_reference is not None else None,
    reference_display_names=REFERENCE_DISPLAY_NAMES,
    training_history=report_training["history"],
    notes=report_notes,
    provenance={"training": report_training["provenance"], "validation_sections": list(VALID_SECTIONS),
                "patch_size": PATCH_SIZE, "smoothing_sigma": SIGMA, "smoothing_neighbors": N_NEIGHBORS},
    crop_selection=REPORT_CROP if report_reference is not None else None,
    crop_name=REPORT_CROP_NAME,
    boundary_grid_size=REPORT_BOUNDARY_GRID,
    boundary_max_distance=REPORT_BOUNDARY_DISTANCE,
)
show_report(report_html)
